In [ ]:
import os
import json
import yaml
import torch
import mmdet
from mmengine.config import Config
from mmengine.runner import Runner
from mmdet.utils import register_all_modules
from mmdet.datasets import CocoDataset

def ensure_dir_exists(path):
    """디렉토리가 존재하는지 확인하고 없으면 생성"""
    if not os.path.exists(path):
        os.makedirs(path)

def load_dataset_config(config_path):
    """YAML에서 데이터셋 설정만 로드"""
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"YAML config file not found at {config_path}")
        
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config

def validate_dataset_paths(dataset_cfg):
    """데이터셋 경로 유효성 검사"""
    data_root = dataset_cfg['dataset']['data_root']
    print(f"\nValidating datasets with data_root: {data_root}")
    
    for split in ['train', 'valid', 'test']:
        print(f"\nValidating {split} dataset:")
        
        # 이미지 디렉토리 확인
        img_path = dataset_cfg['dataset'][split]['img_path']
        print(f"Checking image directory: {img_path}")
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"{split} image directory not found: {img_path}")
        
        # 이미지 파일 확인
        img_files = [f for f in os.listdir(img_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"Found {len(img_files)} image files")
        if not img_files:
            raise ValueError(f"No image files found in {img_path}")
            
        # 어노테이션 파일 확인
        ann_file = dataset_cfg['dataset'][split]['ann_file']
        print(f"Checking annotation file: {ann_file}")
        if not os.path.exists(ann_file):
            raise FileNotFoundError(f"{split} annotation file not found: {ann_file}")
            
        # 어노테이션 파일 내용 확인
        with open(ann_file, 'r', encoding='utf-8') as f:
            try:
                annotations = json.load(f)
                print(f"Annotation file structure:")
                print(f"- Number of images: {len(annotations['images'])}")
                print(f"- Number of annotations: {len(annotations['annotations'])}")
                print(f"- Number of categories: {len(annotations['categories'])}")
                
                # 이미지 ID와 파일 존재 여부 확인
                for img in annotations['images']:
                    img_file = img['file_name']
                    full_path = os.path.join(img_path, img_file)
                    if not os.path.exists(full_path):
                        raise FileNotFoundError(f"Image file not found: {full_path}")
                
                if len(annotations['images']) == 0 or len(annotations['annotations']) == 0:
                    raise ValueError(f"Empty {split} dataset detected")
                    
                # 카테고리 확인
                categories = {cat['id']: cat['name'] for cat in annotations['categories']}
                print(f"- Categories: {categories}")
                
                # 클래스 일치 확인
                config_classes = set(dataset_cfg['dataset']['classes'])
                anno_classes = set(cat['name'] for cat in annotations['categories'])
                if config_classes != anno_classes:
                    print(f"Warning: Config classes ({config_classes}) do not match annotation classes ({anno_classes})")
                
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON format in {ann_file}: {str(e)}")
            except KeyError as e:
                raise ValueError(f"Missing required key in annotation file: {str(e)}")
                
        print(f"{split} dataset validation completed successfully")

def create_dataset_config(dataset_cfg, split='train', pipeline=None):
    """데이터셋 설정 생성"""
    # 절대 경로에서 상대 경로로 변환
    data_root = dataset_cfg['dataset']['data_root']
    img_path = dataset_cfg['dataset'][split]['img_path']
    ann_file = dataset_cfg['dataset'][split]['ann_file']
    
    # 상대 경로 계산
    rel_img_path = os.path.relpath(img_path, data_root)
    rel_ann_file = os.path.relpath(ann_file, data_root)
    
    print(f"\nDataset config for {split}:")
    print(f"data_root: {data_root}")
    print(f"rel_img_path: {rel_img_path}")
    print(f"rel_ann_file: {rel_ann_file}")
    
    # 클래스 이름을 소문자로 통일
    classes = [cls.lower() for cls in dataset_cfg['dataset']['classes']]
    print(f"classes: {classes}")
    
    dataset_dict = dict(
        type='CocoDataset',  # 필수 type 필드 추가
        data_root=data_root,
        ann_file=rel_ann_file,
        data_prefix=dict(img=rel_img_path),
        test_mode=(split != 'train'),
        filter_cfg=dict(filter_empty_gt=True, min_size=32) if split == 'train' else None,
        metainfo=dict(classes=classes),
        pipeline=pipeline
    )
    
    return dataset_dict

def setup_model_config(cfg_data):
    """모델 및 학습 설정"""
    # Pipeline 설정 수정
    train_pipeline = [
        dict(type='LoadImageFromFile'),
        dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
        dict(
            type='CachedMosaic',
            img_scale=(640, 640),
            pad_val=114.0,
            max_cached_images=10,
            random_pop=False),
        dict(
            type='RandomResize',
            scale=(640, 640),
            ratio_range=(0.1, 2.0),
            keep_ratio=True),
        dict(type='RandomCrop', crop_size=(640, 640)),
        dict(type='YOLOXHSVRandomAug'),
        dict(type='RandomFlip', prob=0.5),
        dict(
            type='Pad',
            size=(640, 640),
            pad_val=dict(img=(114, 114, 114))),
        dict(
            type='CachedMixUp',
            img_scale=(640, 640),
            ratio_range=(1.0, 1.0),
            max_cached_images=10,
            random_pop=False,
            pad_val=(114, 114, 114),
            prob=0.5),
        dict(type='FilterAnnotations', min_gt_bbox_wh=(1, 1)),
        dict(type='PackDetInputs')
    ]

    test_pipeline = [
        dict(type='LoadImageFromFile'),
        dict(type='Resize', scale=(640, 640), keep_ratio=True),
        dict(type='Pad', size=(640, 640), pad_val=dict(img=(114, 114, 114))),
        dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
        dict(
            type='PackDetInputs',
            meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape',
                      'scale_factor'))
    ]

    # RTMDet-Ins 설정 파일 경로
    base_config = 'configs/rtmdet/rtmdet-ins_l_8xb32-300e_coco.py'
    if not os.path.exists(base_config):
        base_config = 'C:/Users/AI-LHJ/Desktop/mmdetection-main/configs/rtmdet/rtmdet-ins_l_8xb32-300e_coco.py'
        if not os.path.exists(base_config):
            raise FileNotFoundError("Model config file not found")

    print(f"Loading config from: {base_config}")
    cfg = Config.fromfile(base_config)

    # 모델 가중치 설정
    cfg.load_from = None

    # 데이터셋 설정
    cfg.train_dataloader = dict(
        batch_size=2,
        num_workers=4,
        persistent_workers=True,
        pin_memory=True,  # GPU 전송 최적화
        sampler=dict(type='DefaultSampler', shuffle=True),
        batch_sampler=None,
        dataset=create_dataset_config(cfg_data, 'train', pipeline=train_pipeline)
    )

    cfg.val_dataloader = dict(
        batch_size=1,
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=create_dataset_config(cfg_data, 'valid', pipeline=test_pipeline)
    )

    cfg.test_dataloader = dict(
        batch_size=1,
        num_workers=cfg_data['training']['num_workers'],
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=create_dataset_config(cfg_data, 'test', pipeline=test_pipeline)
    )

    # 평가 설정
    cfg.val_evaluator = dict(
        type='CocoMetric',
        ann_file=cfg_data['dataset']['valid']['ann_file'],
        metric=['segm'],
        classwise=True,
        format_only=False)

    cfg.test_evaluator = dict(
        type='CocoMetric',
        ann_file=cfg_data['dataset']['test']['ann_file'],
        metric=['segm'],
        classwise=True,
        format_only=False)

    # 학습 설정
    cfg.train_cfg = dict(
        type='EpochBasedTrainLoop',
        max_epochs=cfg_data['training']['max_epochs'],
        val_interval=1)

    # 옵티마이저 설정
    cfg.optim_wrapper = dict(
        type='AmpOptimWrapper',
        optimizer=dict(
            type='AdamW',
            lr=cfg_data['training']['lr'],
            weight_decay=0.05),
        clip_grad=None)

    # 학습률 스케줄러
    cfg.param_scheduler = [
        dict(
            type='LinearLR',
            start_factor=0.001,
            by_epoch=False,
            begin=0,
            end=1000),
        dict(
            type='MultiStepLR',
            begin=0,
            end=cfg_data['training']['max_epochs'],
            by_epoch=True,
            milestones=[27, 33],
            gamma=0.1)
    ]

    # 작업 디렉토리 설정
    cfg.work_dir = cfg_data['model']['work_dir']
    ensure_dir_exists(cfg.work_dir)

    # 모델 설정
    num_classes = len(cfg_data['dataset']['classes'])
    if hasattr(cfg.model, 'bbox_head'):
        cfg.model.bbox_head.num_classes = num_classes
    if hasattr(cfg.model, 'mask_head'):
        cfg.model.mask_head.num_classes = num_classes

    return cfg

def save_and_print_metrics(runner):
    """학습 결과 메트릭스를 저장하고 출력"""
    metrics = runner.message_hub.get_info('metrics')
    
    if metrics is None:
        print("No metrics available")
        return
        
    results = {}
    for class_name in ['All'] + runner.train_dataloader.dataset.metainfo['classes']:
        results[class_name] = {
            'precision': 0,
            'recall': 0,
            'mAP50': 0,
            'mAP50-95': 0
        }
    
    # 메트릭스 정보 추출
    for metric_name, value in metrics.items():
        if 'segm' in metric_name:  # 세그멘테이션 메트릭스만 처리
            parts = metric_name.split('/')
            class_name = parts[1] if len(parts) > 1 else 'All'
            
            if class_name in results:
                if 'precision' in metric_name:
                    results[class_name]['precision'] = float(value)
                elif 'recall' in metric_name:
                    results[class_name]['recall'] = float(value)
                elif 'mAP_50' in metric_name:
                    results[class_name]['mAP50'] = float(value)
                elif 'mAP' in metric_name and '50' not in metric_name:
                    results[class_name]['mAP50-95'] = float(value)
    
    # 결과 출력
    print("\nSegmentation Metrics:")
    print(f"{'Class':<10} {'Precision':<10} {'Recall':<10} {'mAP50':<10} {'mAP50-95':<10}")
    print("-" * 60)
    
    for class_name, metrics in results.items():
        print(f"{class_name:<10} {metrics['precision']:<10.3f} {metrics['recall']:<10.3f} "
              f"{metrics['mAP50']:<10.3f} {metrics['mAP50-95']:<10.3f}")
    
    # 결과를 JSON 파일로 저장
    save_path = os.path.join(runner.work_dir, 'segmentation_metrics.json')
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"\nMetrics saved to: {save_path}")

# GPU 메모리 및 사용률 제한 설정 함수 추가
def setup_gpu_limits():
    """GPU 메모리 및 사용률 제한 설정"""
    # GPU 사용률 제한 (예: 90%)
    torch.cuda.set_per_process_memory_fraction(0.90)
    
    # 메모리 할당 전략 설정
    torch.cuda.set_device(0)  # 첫 번째 GPU 사용
    torch.backends.cudnn.benchmark = False  # 벤치마크 비활성화로 메모리 사용 감소
    
    # 캐시 자동 비우기 활성화
    torch.cuda.empty_cache()

def train_model(yaml_config_path):
    """모델 학습 실행"""
    try:
        # GPU 제한 설정 적용
        setup_gpu_limits()

        # .mim 디렉토리 생성
        mim_dir = os.path.join(os.path.dirname(mmdet.__file__), '.mim')
        ensure_dir_exists(mim_dir)
        
        # model-index.yml 파일 생성
        model_index_path = os.path.join(mim_dir, 'model-index.yml')
        if not os.path.exists(model_index_path):
            model_index_content = {
                'Import': [],
                'Collections': []
            }
            with open(model_index_path, 'w') as f:
                yaml.dump(model_index_content, f)
        
        # 모든 모듈 등록
        register_all_modules()
        
        # 설정 로드 및 검증
        cfg_data = load_dataset_config(yaml_config_path)
        validate_dataset_paths(cfg_data)
        cfg = setup_model_config(cfg_data)
        
        # Runner 생성 및 학습
        runner = Runner.from_cfg(cfg)
        
        try:
            runner.train()
        except Exception as e:
            if 'HTTP Error 404' in str(e):
                print("체크포인트 다운로드 실패. ResNet-50 기본 가중치를 사용합니다.")
                cfg.load_from = None
                runner = Runner.from_cfg(cfg)
                runner.train()
            else:
                raise
        
        # 테스트 실행 및 메트릭스 저장
        runner.test()
        save_and_print_metrics(runner)
        
    except Exception as e:
        print(f"Error during training: {str(e)}")
        raise


if __name__ == '__main__':
    yaml_config_path = 'C:/Users/AI-LHJ/Desktop/json dataset/dataset.yaml'  # YAML 파일 경로
    train_model(yaml_config_path)